# Readmission ranking under a fixed outreach budget (public data)

Compare ranking by prior inpatient visits with a gradient-boosted ranker using eight encounter measurements. Hold out 25% of people in each split. Measure the fraction of positive encounters captured in the top 10% and the positive fraction within that budget. Use the UCI Diabetes 130-US Hospitals encounter cohort, filtering discharge categories for hospice or death. Split by patient identifier. The positive outcome is a recorded readmission within 30 days. The experiment tests prioritization and does not estimate whether outreach prevents readmission.

In [ ]:
from pathlib import Path
import subprocess, sys
root = Path.cwd() / 'capacity-readmission-ranking-real-data-run'
root.mkdir(exist_ok=True)
subprocess.run([sys.executable,'-m','pip','install',*['numpy==2.2.6', 'scipy==1.15.3', 'scikit-learn==1.7.2', 'joblib==1.6.0']],check=True)

In [ ]:
(root / 'prepare.py').write_text('import os\nfrom pathlib import Path\nfrom urllib.request import urlretrieve\n\ndata = Path(os.environ.get("PORTFOLIO_DATA", Path(__file__).resolve().parent / "data"))\ndata.mkdir(parents=True, exist_ok=True)\ntarget = data / "diabetes130.zip"\nif not target.exists():\n    temp = target.with_suffix(".part")\n    urlretrieve(\n        "https://archive.ics.uci.edu/static/public/296/diabetes+130-us+hospitals+for+years+1999-2008.zip",\n        temp,\n    )\n    temp.replace(target)\n',encoding='utf8')
subprocess.run([sys.executable,'prepare.py'],cwd=root,check=True)

In [ ]:
(root / 'train.py').write_text('import json\nimport os\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nfrom sklearn.metrics import (\n    average_precision_score,\n    roc_auc_score,\n)\n\nROOT = Path(__file__).resolve().parent\nDATA = Path(os.environ.get("PORTFOLIO_DATA", ROOT / "data"))\nSEEDS = [13, 37, 61]\n\n\ndef save(obj, name):\n    folder = ROOT / "checkpoints"\n    folder.mkdir(exist_ok=True)\n    joblib.dump(obj, folder / (name + ".joblib"), compress=3)\n\n\ndef scores(y, p):\n    return {\n        "auroc": float(roc_auc_score(y, p)),\n        "average_precision": float(average_precision_score(y, p)),\n    }\n\n\nimport csv\nimport io\nimport zipfile\n\n\ndef dataset(seed):\n    x = []\n    y = []\n    groups = []\n    keys = [\n        "number_inpatient",\n        "number_emergency",\n        "number_outpatient",\n        "time_in_hospital",\n        "num_lab_procedures",\n        "num_procedures",\n        "num_medications",\n        "number_diagnoses",\n    ]\n    with zipfile.ZipFile(DATA / "diabetes130.zip") as z:\n        with z.open("diabetic_data.csv") as raw:\n            for row in csv.DictReader(io.TextIOWrapper(raw)):\n                if row["discharge_disposition_id"] in {\n                    "11",\n                    "13",\n                    "14",\n                    "19",\n                    "20",\n                    "21",\n                }:\n                    continue\n                x.append([float(row[k]) for k in keys])\n                y.append(int(row["readmitted"] == "<30"))\n                groups.append(int(row["patient_nbr"]))\n    return np.array(x), np.array(y), np.array(groups)\n\n\nfrom sklearn.ensemble import HistGradientBoostingClassifier\nfrom sklearn.model_selection import GroupShuffleSplit\n\n\ndef run(seed):\n    x, y, groups = dataset(seed)\n    train, test = next(\n        GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed).split(\n            x, y, groups\n        )\n    )\n    assert not set(groups[train]) & set(groups[test])\n    model = HistGradientBoostingClassifier(\n        max_iter=130, max_leaf_nodes=15, l2_regularization=2, random_state=seed\n    ).fit(x[train], y[train])\n    save(model, f"ranker-{seed}")\n    out = {}\n    for method, p in [\n        ("prior_inpatient_visits", x[test, 0]),\n        ("learned_priority", model.predict_proba(x[test])[:, 1]),\n    ]:\n        order = np.argsort(-p, kind="stable")\n        eligible = order[: int(0.10 * len(test))]\n        out[method] = {\n            **scores(y[test], p),\n            "recall_at_budget": float(y[test][eligible].sum() / y[test].sum()),\n            "precision_at_budget": float(y[test][eligible].mean()),\n        }\n    return out\n\n\nif __name__ == "__main__":\n    runs = {s: run(s) for s in SEEDS}\n    first = next(iter(runs.values()))\n    means = {\n        k: {m: float(np.mean([r[k][m] for r in runs.values()])) for m in v}\n        for k, v in first.items()\n    }\n    (ROOT / "results.json").write_text(\n        json.dumps({"seeds": SEEDS, "runs": runs, "means": means}, indent=2)\n    )\n    print(means)\n',encoding='utf8')
subprocess.run([sys.executable,'train.py'],cwd=root,check=True)

In [ ]:
print((root/'results.json').read_text())